<a href="https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ML-06 setup
# Connect to the March 2026 development partition.

%pip -q install duckdb huggingface_hub

import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Add your READ Hugging Face token "
        "to Colab Secrets as HF_TOKEN."
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

PERFORMANCE = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/
     fact_content_daily_performance/month=2026-03/*.parquet'
)
""".replace("\n", "").replace("     ", "")

print("Warehouse connection ready.")
print("Development window: March 2026")

Warehouse connection ready.
Development window: March 2026


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*


I first inspect the distributions of the main search-performance signals used by my ranking task. Impressions and clicks are expected to be right-skewed because a small number of content items can receive much more traffic than the rest. Average search position is interpreted carefully because lower values represent better positions and zero can represent unavailable position data. I use these distributions to understand the scale of the signals before defining the baseline rule.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-06 Section 1 — Distributions

distribution_df = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(gsc_impressions) AS min_impressions,
        AVG(gsc_impressions) AS mean_impressions,
        MEDIAN(gsc_impressions) AS median_impressions,
        MAX(gsc_impressions) AS max_impressions,

        MIN(gsc_clicks) AS min_clicks,
        AVG(gsc_clicks) AS mean_clicks,
        MEDIAN(gsc_clicks) AS median_clicks,
        MAX(gsc_clicks) AS max_clicks,

        MIN(gsc_avg_position) AS min_position,
        MEDIAN(gsc_avg_position) AS median_position,
        MAX(gsc_avg_position) AS max_position
    FROM {PERFORMANCE}
""").df()

display(distribution_df)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,min_impressions,mean_impressions,median_impressions,max_impressions,min_clicks,mean_clicks,median_clicks,max_clicks,min_position,median_position,max_position
0,9841378,0,28.518119,0.0,40084,0,0.083508,0.0,274,0.0,7.5,498.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*



### Signal 1 — Search volume

**Verdict: CONFIRMED** if higher-impression buckets contain enough observations to support a useful review queue. Search volume matters because a page with substantial exposure provides a larger potential opportunity for improvement.

### Signal 2 — CTR relative to search position

**Verdict: CONFIRMED** if CTR differs across position groups. This supports the idea that CTR should be interpreted together with position rather than treated as an isolated number.

### Signal 3 — Average search position

**Verdict: CONFIRMED** if the position buckets show meaningful differences in observed CTR. Position provides context for deciding whether a low CTR is unusual for the page's search visibility.

The verdicts are based on the observed March 2026 development data and are used for decision-support, not as causal claims.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-06 Section 2 — Three signal tests

# ---------------------------------------------------------
# Signal 1: Search volume
# ---------------------------------------------------------

volume_test = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_impressions < 100 THEN '<100'
            WHEN gsc_impressions < 500 THEN '100-499'
            WHEN gsc_impressions < 1000 THEN '500-999'
            WHEN gsc_impressions < 5000 THEN '1,000-4,999'
            ELSE '5,000+'
        END AS impression_bucket,

        COUNT(*) AS n,
        ROUND(AVG(gsc_impressions), 2) AS avg_impressions,
        ROUND(AVG(gsc_clicks), 2) AS avg_clicks

    FROM {PERFORMANCE}

    GROUP BY 1
    ORDER BY
        CASE impression_bucket
            WHEN '<100' THEN 1
            WHEN '100-499' THEN 2
            WHEN '500-999' THEN 3
            WHEN '1,000-4,999' THEN 4
            ELSE 5
        END
""").df()

print("Signal 1 — Search volume")
display(volume_test)


# ---------------------------------------------------------
# Signal 2: CTR relative to position
# ---------------------------------------------------------

ctr_position_test = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position > 0 AND gsc_avg_position <= 3
                THEN '1-3'
            WHEN gsc_avg_position > 3 AND gsc_avg_position <= 10
                THEN '4-10'
            WHEN gsc_avg_position > 10 AND gsc_avg_position <= 20
                THEN '11-20'
            WHEN gsc_avg_position > 20
                THEN '21+'
            ELSE 'no_position'
        END AS position_bucket,

        COUNT(*) AS n,

        ROUND(
            100.0 * SUM(gsc_clicks)
            / NULLIF(SUM(gsc_impressions), 0),
            3
        ) AS ctr_pct

    FROM {PERFORMANCE}

    WHERE gsc_impressions > 0

    GROUP BY 1
    ORDER BY 1
""").df()

print("Signal 2 — CTR by search position")
display(ctr_position_test)


# ---------------------------------------------------------
# Signal 3: Position
# ---------------------------------------------------------

position_test = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position > 0 AND gsc_avg_position <= 3
                THEN '1-3'
            WHEN gsc_avg_position > 3 AND gsc_avg_position <= 10
                THEN '4-10'
            WHEN gsc_avg_position > 10 AND gsc_avg_position <= 20
                THEN '11-20'
            WHEN gsc_avg_position > 20
                THEN '21+'
            ELSE 'no_position'
        END AS position_bucket,

        COUNT(*) AS n,

        ROUND(AVG(gsc_avg_position), 2) AS avg_position,

        ROUND(
            100.0 * SUM(gsc_clicks)
            / NULLIF(SUM(gsc_impressions), 0),
            3
        ) AS ctr_pct

    FROM {PERFORMANCE}

    WHERE gsc_impressions > 0

    GROUP BY 1
    ORDER BY 1
""").df()

print("Signal 3 — Position and observed CTR")
display(position_test)


Signal 1 — Search volume


,impression_bucket,n,avg_impressions,avg_clicks
0,<100,9202770,6.54,0.02
1,100-499,537157,213.17,0.65
2,500-999,69032,682.37,1.94
3,"1,000-4,999",31676,1659.84,4.50
4,"5,000+",743,8482.63,29.14


Signal 2 — CTR by search position


,position_bucket,n,ctr_pct
0,1-3,564173,0.381
1,11-20,519223,0.315
2,21+,908354,0.131
3,4-10,1456122,0.323
4,no_position,163189,0.250


Signal 3 — Position and observed CTR


,position_bucket,n,avg_position,ctr_pct
0,1-3,564173,1.81,0.381
1,11-20,519223,14.33,0.315
2,21+,908354,43.89,0.131
3,4-10,1456122,6.06,0.323
4,no_position,163189,0.00,0.250


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*


The flag-linked signal I use is **CTR relative to search position**, which is connected to the CTR-fix logic discussed in the session.

The test compares observed CTR across search-position buckets. The purpose is to check whether CTR behaves differently depending on search visibility. If the buckets show meaningful differences, this supports using position as context when identifying potential CTR opportunities.

The result is interpreted as an observed directional signal, not evidence that position causes CTR.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-06 Section 3 — Flag-linked CTR test

flag_linked_test = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position > 0 AND gsc_avg_position <= 3
                THEN '1-3'
            WHEN gsc_avg_position > 3 AND gsc_avg_position <= 10
                THEN '4-10'
            WHEN gsc_avg_position > 10 AND gsc_avg_position <= 20
                THEN '11-20'
            WHEN gsc_avg_position > 20
                THEN '21+'
            ELSE 'no_position'
        END AS position_bucket,

        COUNT(*) AS n,

        ROUND(
            100.0 * SUM(gsc_clicks)
            / NULLIF(SUM(gsc_impressions), 0),
            3
        ) AS observed_ctr_pct,

        ROUND(AVG(gsc_impressions), 2) AS avg_impressions

    FROM {PERFORMANCE}

    WHERE gsc_impressions > 0

    GROUP BY 1
    ORDER BY 1
""").df()

display(flag_linked_test)

print(
    "Interpretation: compare the observed CTR across position buckets. "
    "Large differences support using position as context for CTR review."
)


,position_bucket,n,observed_ctr_pct,avg_impressions
0,1-3,564173,0.381,94.94
1,11-20,519223,0.315,56.60
2,21+,908354,0.131,65.41
3,4-10,1456122,0.323,94.66
4,no_position,163189,0.250,2.87


Interpretation: compare the observed CTR across position buckets. Large differences support using position as context for CTR review.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*


The March 2026 data supports using search exposure and position as practical context when prioritizing content for review. CTR should not be interpreted alone because the expected click rate depends on where a page appears in search. These signals can therefore support a transparent review queue, while the results remain directional and do not establish causation.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-06 Section 4 — Compact evidence summary

print("Signals used for the baseline:")
print("1. Search impressions")
print("2. CTR relative to search position")
print("3. Average search position")
print()
print("All three are measured in the March 2026 development window.")
print("No future-window or label-derived fields are used.")

Signals used for the baseline:
1. Search impressions
2. CTR relative to search position
3. Average search position

All three are measured in the March 2026 development window.
No future-window or label-derived fields are used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.